# UC ethnicity outcomes, 2017–2025

Question: Among UC freshman applicants from 2017–2025, how did application share, admission rate, and enrollment yield change across reported race and ethnicity groups, and how did those patterns differ across campuses and years?

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [Path('../Data/uc_admissions_summary_by_ethnicity.csv'), Path('Data/uc_admissions_summary_by_ethnicity.csv')]
data_path = next(path for path in candidates if path.exists())
raw = pd.read_csv(data_path)
raw.shape, raw.columns.tolist()

In [ ]:
key = ['entrant_level', 'campus', 'fall_term', 'ethnicity', 'count_type']
assert not raw.duplicated(key).any()
metrics = (raw.pivot(index=['entrant_level','campus','fall_term','ethnicity'], columns='count_type', values='n')
             .reset_index().rename(columns={'App':'applicants','Adm':'admits','Enr':'enrollees'}))
metrics['application_share'] = metrics['applicants'] / metrics.groupby(['entrant_level','campus','fall_term'])['applicants'].transform('sum')
metrics['admission_rate'] = metrics['admits'] / metrics['applicants']
metrics['yield_rate'] = metrics['enrollees'] / metrics['admits']
metrics.head()

In [ ]:
systemwide = metrics[(metrics.entrant_level == 'freshman') & (metrics.campus == 'Systemwide')]
changes = {}
for metric in ['application_share', 'admission_rate', 'yield_rate']:
    wide = systemwide.pivot(index='ethnicity', columns='fall_term', values=metric)
    changes[metric] = ((wide[2025] - wide[2017]) * 100).sort_values()
    print('\n', metric, '\n', changes[metric].round(2))

In [ ]:
latest = systemwide[systemwide.fall_term == 2025]
applicants = latest.applicants.sum()
admits = latest.admits.sum()
enrollees = latest.enrollees.sum()
summary = {'applicants': int(applicants), 'admits': int(admits), 'enrollees': int(enrollees), 'admission_rate': admits/applicants, 'yield': enrollees/admits}
summary

In [ ]:
assert round(changes['application_share'].loc['Asian'], 2) == 3.42
assert round(changes['application_share'].loc['White'], 2) == -4.69
assert round(changes['admission_rate'].loc['Pacific Islander'], 2) == 22.93
assert round(changes['yield_rate'].loc['Hispanic/Latino(a)'], 2) == -13.55
assert summary['applicants'] == 205389 and summary['admits'] == 148676 and summary['enrollees'] == 52609
print('Headline findings reproduced.')

In [ ]:
systemwide.pivot(index='fall_term', columns='ethnicity', values='application_share').plot.area(figsize=(12, 6), title='Systemwide freshman application composition')